In [1]:
import re
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer

pd.set_option("display.max_colwidth", 100)

df = pd.read_csv("../data/reviews_labeled.csv")
print(df.shape)

(6965, 18)


In [2]:
keep_words = {"not", "no", "nor", "never", "cannot", "very", "too", "least", "less", "more", "most"}
stop_words = ENGLISH_STOP_WORDS - keep_words

print("Stopwords used:", len(stop_words))
print("Examples:", sorted(stop_words)[:15])

Stopwords used: 307
Examples: ['a', 'about', 'above', 'across', 'after', 'afterwards', 'again', 'against', 'all', 'almost', 'alone', 'along', 'already', 'also', 'although']


In [3]:
def clean_text(text):
    text = text.lower()                         # 1. all lowercase
    text = re.sub(r"[^a-z\s]", " ", text)       # 2. keep only letters and spaces
    words = [w for w in text.split()            # 3. split into words
             if w not in stop_words and len(w) > 1]   # 4. drop stopwords and single letters
    return " ".join(words)                      # 5. join back into one string

# test it on two examples
for s in ["The Chair is NOT good!! Delivery took 10 days.",
          "Very poor quality table. Still waiting for a refund."]:
    print(s, "->", clean_text(s))

The Chair is NOT good!! Delivery took 10 days. -> chair not good delivery took days
Very poor quality table. Still waiting for a refund. -> very poor quality table waiting refund


In [4]:
df["Clean Text"] = df["Review Text"].apply(clean_text)
df[["Review Text", "Clean Text"]].sample(6, random_state=2)

,Review Text,Clean Text
4371,Very happy with this bookcase.,very happy bookcase
5172,Very happy with this machine!!,very happy machine
1852,The storage box is sturdy and well made. Good value for money.,storage box sturdy good value money
5665,Very happy with this accessory. Will definitely order again. Installation was easy.,very happy accessory definitely order installation easy
1526,Quality of the copier is average. Packaging was okay.,quality copier average packaging okay
6521,Good.,good


In [5]:
empty = (df["Clean Text"].str.strip() == "").sum()
print("Empty after cleaning:", empty)

df = df[df["Clean Text"].str.strip() != ""]
print("Shape:", df.shape)

Empty after cleaning: 17
Shape: (6948, 19)


In [6]:
before = df["Review Text"].str.split().str.len().mean()
after = df["Clean Text"].str.split().str.len().mean()
print(f"Average words before: {before:.1f} | after: {after:.1f}")

Average words before: 9.9 | after: 6.3


In [7]:
for s in ["Positive", "Negative"]:
    words = " ".join(df[df["Sentiment"] == s]["Clean Text"]).split()
    top = Counter(words).most_common(10)
    print(f"\n{s}:", top)


Positive: [('very', 992), ('good', 771), ('money', 690), ('quality', 573), ('delivery', 549), ('packaging', 499), ('value', 412), ('stars', 367), ('overall', 364), ('smooth', 363)]

Negative: [('not', 708), ('delivery', 454), ('week', 350), ('very', 304), ('took', 278), ('more', 276), ('disappointed', 219), ('customer', 179), ('care', 178), ('price', 158)]


In [8]:
examples = ["chair good quality", "chair not good", "table damaged late delivery"]

demo = TfidfVectorizer()
matrix = demo.fit_transform(examples)

pd.DataFrame(matrix.toarray(), columns=demo.get_feature_names_out(), index=examples).round(2)

,chair,damaged,delivery,good,late,not,quality,table
chair good quality,0.52,0.0,0.0,0.52,0.0,0.00,0.68,0.0
chair not good,0.52,0.0,0.0,0.52,0.0,0.68,0.00,0.0
table damaged late delivery,0.00,0.5,0.5,0.00,0.5,0.00,0.00,0.5


In [9]:
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
X = tfidf.fit_transform(df["Clean Text"])

print("Shape:", X.shape)
print("Some features:", list(tfidf.get_feature_names_out()[:5]))
print("Word pairs with 'not':", [f for f in tfidf.get_feature_names_out() if f.startswith("not ")][:8])

Shape: (6948, 1680)
Some features: ['acceptable', 'acceptable alright', 'acceptable buy', 'acceptable customer', 'acceptable delivery']
Word pairs with 'not': ['not bad', 'not buy', 'not good', 'not great', 'not happy', 'not recommend', 'not respond', 'not shown']


In [10]:
df.to_csv("../data/reviews_clean.csv", index=False)
print("Saved:", df.shape)

Saved: (6948, 19)
